# Module 5: Multi-Agent Systems & Agentic RAG

**Day 4 — Agents, LangGraph & MCP**

## What you will learn
- Supervisor pattern: one coordinator, many specialists
- Subgraph composition in LangGraph
- Agentic RAG: retrieve → grade → rewrite → generate loop
- Agent types: ReAct, Plan-and-Execute, Agentic RAG

## 1. Simplest Multi-Agent: Two Functions, One Supervisor

In [ ]:
# Two specialist functions + one supervisor — no framework
def researcher(topic: str) -> str:
    return f"Research on '{topic}': [finding 1, finding 2, finding 3]"

def writer(topic: str, research: str) -> str:
    return f"## {topic.title()}\n\nBased on: {research[:50]}\n\nConclusion: important topic."

def supervisor(task: str) -> str:
    print(f"[supervisor] starting task: {task}")
    findings = researcher(task)
    print(f"[researcher] {findings[:50]}")
    article  = writer(task, findings)
    print(f"[writer]    generated {len(article)} chars")
    return article

print(supervisor("renewable energy in India"))

## 2. Agentic RAG — The Loop Pattern

Regular RAG: retrieve once → generate.
**Agentic RAG**: retrieve → grade quality → if poor, rewrite query → retrieve again → generate.

```
Query → Retrieve → Grade?
                  ↓ BAD → Rewrite → Retrieve (max 2 retries)
                  ↓ GOOD
               Generate answer
```

In [ ]:
def retrieve(query: str, docs: list) -> list:
    return [d for d in docs if any(w in d.lower() for w in query.lower().split())]

def grade(docs: list) -> bool:
    return len(docs) > 0

def rewrite(query: str) -> str:
    return query + " tutorial guide overview"

def generate(query: str, docs: list) -> str:
    return f"Answer to '{query}': {docs[0][:80]}..."

def agentic_rag(query: str, docs: list, max_iter: int = 2) -> str:
    for i in range(max_iter + 1):
        retrieved = retrieve(query, docs)
        print(f"  iter {i}: '{query[:40]}' → {len(retrieved)} docs")
        if grade(retrieved):
            return generate(query, retrieved)
        query = rewrite(query)
    return "No relevant answer found."

docs = [
    "LangGraph is a framework for building stateful multi-agent workflows",
    "Python is a widely-used programming language for data science",
    "Agentic RAG combines retrieval with iterative query refinement",
]
print(agentic_rag("langgraph", docs))
print()
print(agentic_rag("xyz123", docs))

## 3. Agent Type Comparison

| Agent | Strategy | Best for |
|-------|----------|----------|
| **ReAct** | Reason-Act-Observe loop | Dynamic tasks with unknowns |
| **Plan-and-Execute** | Plan all steps upfront | Long structured pipelines |
| **Agentic RAG** | Retrieve-Grade-Rewrite loop | High-quality Q&A |

**ReAct** (Yao et al. 2022): the LLM alternates between thinking and acting, adjusting based on tool results.

**Plan-and-Execute**: useful when you know the task structure upfront (e.g., "research X, write Y, format Z").

## 4. Using day4 modules

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from day4.multi_agent import AgentType
print("Agent types:")
for a in AgentType:
    print(f"  {a.name}: {a.value}")

In [ ]:
from day4.multi_agent import build_research_subgraph, build_writer_subgraph

def mock_research(topic: str) -> str:
    return f"Research: '{topic}' uses attention mechanisms, pre-trained on large corpora."

def mock_write(topic: str, findings: str) -> str:
    return f"**{topic.title()}**\n\n{findings[:100]}\n\nKey takeaway: transformative technology."

research_g = build_research_subgraph(mock_research)
writer_g   = build_writer_subgraph(mock_write)
print("Research subgraph:", type(research_g).__name__)
print("Writer subgraph:  ", type(writer_g).__name__)

In [ ]:
from day4.multi_agent import build_research_team
from langchain_core.messages import HumanMessage

team   = build_research_team(research_fn=mock_research, write_fn=mock_write)
result = team.invoke({"messages": [HumanMessage("Large language models")]})
print(result.get("final_output", "(no output)"))

In [ ]:
from day4.multi_agent import build_agentic_rag_graph

RAG_DOCS = [
    "LangGraph supports stateful workflows with MemorySaver checkpointing",
    "Multi-agent systems use supervisor patterns for task coordination",
    "Agentic RAG retrieves, grades, and rewrites queries iteratively",
]

def mock_retrieve(q): return [d for d in RAG_DOCS if any(w in d.lower() for w in q.split())]
def mock_gen(q, docs): return f"Answer: {docs[0][:80] if docs else 'No docs'}..."

rag = build_agentic_rag_graph(mock_retrieve, mock_gen)
s   = rag.invoke({"query": "langgraph checkpointing", "documents": [], "answer": "", "needs_rewrite": False, "iteration": 0})
print("Query: ", s["query"])
print("Answer:", s["answer"])